# Ecommerce Customer Churn Analysis and Prediction
## Etapa 01: Análise Exploratória de Dados (EDA)

**Objetivo:** Compreender a estrutura dos dados brutos vindos do BigQuery, diagnosticar a qualidade dos dados (nulos e inconsistências) e levantar as primeiras hipóteses de negócio sobre o Churn.

### Dicionário de Dados:

* **CustomerID:** Identificador único do cliente.
* **Churn:** Indicador de cancelamento/abandono (1 = Sim, 0 = Não).
* **Tenure:** Tempo de permanência (em meses) do cliente na plataforma.
* **PreferredLoginDevice:** Dispositivo de login preferido do cliente (ex: Mobile Phone, Computer).
* **CityTier:** Nível/Classificação de relevância econômica da cidade do cliente (1, 2 ou 3).
* **WarehouseToHome:** Distância entre o centro de distribuição (armazém) e a residência do cliente.
* **PreferredPaymentMode:** Método de pagamento preferido do cliente.
* **Gender:** Gênero do cliente.
* **HourSpendOnApp:** Quantidade de horas que o cliente passa navegando no aplicativo ou site.
* **NumberOfDeviceRegistered:** Número total de dispositivos cadastrados por aquele cliente.
* **PreferedOrderCat:** Categoria de produtos favorita ou mais comprada pelo cliente.
* **SatisfactionScore:** Pontuação de satisfação do cliente com o serviço (de 1 a 5).
* **MaritalStatus:** Estado civil do cliente.
* **NumberOfAddress:** Quantidade total de endereços de entrega cadastrados pelo cliente.
* **Complain:** Indicador se o cliente registrou alguma reclamação no último mês (1 = Sim, 0 = Não).
* **OrderAmountHikeFromlastYear:** Porcentagem de aumento no valor gasto em compras em relação ao ano passado.
* **CouponUsed:** Quantidade total de cupons de desconto utilizados pelo cliente no último mês.
* **OrderCount:** Número total de pedidos realizados pelo cliente no último mês.
* **DaySinceLastOrder:** Quantidade de dias decorridos desde a última compra realizada pelo cliente.
* **CashbackAmount:** Valor médio de cashback recebido pelo cliente no último mês.

In [1]:
import os
import pandas as pd
import pandas_gbq


os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'credenciais_google.json'


PROJECT_ID = 'groovy-treat-471720-n1'
QUERY="""
    SELECT * FROM `groovy-treat-471720-n1.1.EComm`
"""

print("Iniciando a extração dos dados brutos do BigQuery...")

# 3. Executa a query e armazena os dados em um DataFrame do Pandas
df_bruto = pandas_gbq.read_gbq(QUERY, project_id=PROJECT_ID)

Iniciando a extração dos dados brutos do BigQuery...


d:\Cursos-Em-Geral\data_analyst\projeto4_bigquery\venv\Lib\site-packages\google\cloud\bigquery\table.py:2806: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


In [2]:
df_bruto.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50354,0,<NA>,Phone,3,19,CC,Male,2,3,Mobile,3,Divorced,2,0,11,1,1,6,121
1,51173,1,<NA>,Phone,2,26,CC,Female,3,3,Mobile,4,Married,1,1,11,0,1,4,121
2,51824,0,<NA>,Phone,3,19,CC,Male,2,3,Mobile,3,Married,2,0,11,1,1,6,121
3,52643,1,<NA>,Phone,2,26,CC,Female,3,3,Mobile,4,Married,1,1,11,0,1,4,121
4,50797,0,<NA>,Computer,1,6,CC,Male,2,3,Mobile,3,Single,1,1,11,0,1,2,122


## Identificando Inconsistencias

In [3]:
df_bruto.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   CustomerID                   5630 non-null   Int64 
 1   Churn                        5630 non-null   Int64 
 2   Tenure                       5366 non-null   Int64 
 3   PreferredLoginDevice         5630 non-null   object
 4   CityTier                     5630 non-null   Int64 
 5   WarehouseToHome              5379 non-null   Int64 
 6   PreferredPaymentMode         5630 non-null   object
 7   Gender                       5630 non-null   object
 8   HourSpendOnApp               5375 non-null   Int64 
 9   NumberOfDeviceRegistered     5630 non-null   Int64 
 10  PreferedOrderCat             5630 non-null   object
 11  SatisfactionScore            5630 non-null   Int64 
 12  MaritalStatus                5630 non-null   object
 13  NumberOfAddress              5630

In [4]:
df_bruto.isnull().sum().sort_values(ascending=False)

DaySinceLastOrder              307
OrderAmountHikeFromlastYear    265
Tenure                         264
OrderCount                     258
CouponUsed                     256
HourSpendOnApp                 255
WarehouseToHome                251
CustomerID                       0
PreferredLoginDevice             0
Churn                            0
PreferredPaymentMode             0
CityTier                         0
SatisfactionScore                0
PreferedOrderCat                 0
NumberOfDeviceRegistered         0
Gender                           0
Complain                         0
NumberOfAddress                  0
MaritalStatus                    0
CashbackAmount                   0
dtype: int64

In [5]:
colunas_categorica = df_bruto.select_dtypes(include='object').columns.to_list()
colunas_categorica

['PreferredLoginDevice',
 'PreferredPaymentMode',
 'Gender',
 'PreferedOrderCat',
 'MaritalStatus']

In [6]:
for coluna in colunas_categorica:
    print(f"\nValores unico na coluna {coluna}:")
    print(df_bruto[coluna].value_counts(dropna=False))


Valores unico na coluna PreferredLoginDevice:
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

Valores unico na coluna PreferredPaymentMode:
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

Valores unico na coluna Gender:
Gender
Male      3384
Female    2246
Name: count, dtype: int64

Valores unico na coluna PreferedOrderCat:
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

Valores unico na coluna MaritalStatus:
MaritalStatus
Married     2986
Single      1796
Divorced     848
Name: count, dtype: int64


- Até aqui, identificamos 7 colunas com valores nulos, e algumas colunas categóricas com valores redundantes, isso é, que tão falando a mesma informação (PreferedOrderCat, PreferredPaymentMode, PreferredLoginDevice)

## Tratando Inconsistencias

In [7]:
lista_dNulos = ['DaySinceLastOrder', 'Tenure', 'OrderCount', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'HourSpendOnApp', 'WarehouseToHome']

for coluna in lista_dNulos:
    grupo_valido = df_bruto[df_bruto[coluna].notnull()]
    print(f"--- Taxa de Churn no Grupo com {coluna} PREENCHIDO ---")
    print(grupo_valido['Churn'].value_counts(normalize=True) * 100)
    
    grupo_nulo = df_bruto[df_bruto[coluna].isnull()]
    print(f"\n--- Taxa de Churn no Grupo com {coluna} NULO ---")
    print(grupo_nulo['Churn'].value_counts(normalize=True) * 100, "\n\n\n")

--- Taxa de Churn no Grupo com DaySinceLastOrder PREENCHIDO ---
Churn
0    83.20496
1    16.79504
Name: proportion, dtype: Float64

--- Taxa de Churn no Grupo com DaySinceLastOrder NULO ---
Churn
0    82.410423
1    17.589577
Name: proportion, dtype: Float64 



--- Taxa de Churn no Grupo com Tenure PREENCHIDO ---
Churn
0    83.842713
1    16.157287
Name: proportion, dtype: Float64

--- Taxa de Churn no Grupo com Tenure NULO ---
Churn
0    69.318182
1    30.681818
Name: proportion, dtype: Float64 



--- Taxa de Churn no Grupo com OrderCount PREENCHIDO ---
Churn
0    82.688012
1    17.311988
Name: proportion, dtype: Float64

--- Taxa de Churn no Grupo com OrderCount NULO ---
Churn
0    93.023256
1     6.976744
Name: proportion, dtype: Float64 



--- Taxa de Churn no Grupo com OrderAmountHikeFromlastYear PREENCHIDO ---
Churn
0    82.590867
1    17.409133
Name: proportion, dtype: Float64

--- Taxa de Churn no Grupo com OrderAmountHikeFromlastYear NULO ---
Churn
0    94.716981
1     5.28

## Diagnóstico Estratégico: Análise de Nulos vs. Churn

Após mapear o comportamento da taxa de Churn em relação aos dados faltantes, identificamos que a ausência de informações **não é aleatória** na maioria dos casos. O simples fato de um dado estar ausente funciona como um forte indicador de comportamento de negócio.

Podemos dividir as 7 colunas com valores nulos em **3 Grupos Distintos**:

---

### Grupo 1 — Nulos com Alto Impacto (Alerta Vermelho)
Variáveis onde a ausência do dado está diretamente associada a um aumento drástico na taxa de cancelamento. Esses nulos indicam falhas no processo de *onboarding*, problemas de coleta ou completo desengajamento do usuário. **Aqui, a estratégia ideal é criar uma Flag (`_isNull`) para não mascarar o risco.**

* **`WarehouseToHome` (Distância Logística)**
    * *Preenchido:* 16.0% | *Nulo:* **33.4%** de Churn (Mais que o dobro da base)
    * **Insight:** A ausência de informação logística indica um problema operacional claro (erro de cadastro, entrega não rastreada ou cliente pouco ativo). Falta de informação logística está fortemente associada a um alto índice de cancelamento.
* **`Tenure` (Tempo de Permanência)**
    * *Preenchido:* 16.1% | *Nulo:* **30.6%** de Churn
    * **Insight:** Clientes com Tenure não registrado apresentam quase o dobro de Churn da base preenchida. Indica falhas graves na retenção inicial, no processo de cadastro ou perfis que nunca engajaram. 
* **`HourSpendOnApp` (Horas Gastas no App)**
    * *Preenchido:* 16.5% | *Nulo:* **22.7%** de Churn
    * **Insight:** Ausência de uso gera ausência de dado. Clientes que não engajam no aplicativo têm uma probabilidade significativamente maior de Churn.

---

### Grupo 2 — Nulos com Baixo Churn (Estágio do Cliente)
Variáveis ligadas a histórico de compra e benefícios, onde o nulo apresenta um comportamento contraintuitivo: a taxa de Churn cai drasticamente. **Interpretação:** Nulo aqui provavelmente significa **Zero** (cliente novo que ainda não gerou histórico).

* **`CouponUsed` (Cupons Utilizados)**
    * *Preenchido:* 17.4% | *Nulo:* **3.1%** de Churn
    * **Insight:** Quem usa cupom churna mais, sugerindo um comportamento oportunista (caçadores de promoção). Quem tem nulo provavelmente nunca usou cupons e é um cliente mais estável ou dormente.
* **`OrderAmountHikeFromlastYear` (Aumento no Valor do Pedido)**
    * *Preenchido:* 17.4% | *Nulo:* **5.2%** de Churn
    * **Insight:** Nulo indica clientes sem histórico do ano passado (clientes novos ou inativos no passado recente).
* **`OrderCount` (Contagem de Pedidos)**
    * *Preenchido:* 17.3% | *Nulo:* **6.9%** de Churn
    * **Insight:** Indica clientes que ainda não tiveram tempo ou chance de realizar novos pedidos e, consequentemente, ainda não tiveram a oportunidade de passar pela janela de Churn (viés temporal).

---

### Grupo 3 — Neutro (Ruído Técnico)
Variáveis onde a ausência do dado não altera em nada o comportamento de negócio. **Pode receber imputação simples (mediana) sem riscos.**

* **`DaySinceLastOrder` (Dias desde a última compra)**
    * *Preenchido:* 16.7% | *Nulo:* **17.5%** de Churn
    * **Insight:** Diferença mínima e irrelevante. Trata-se de uma falha de coleta ou sincronização puramente aleatória.


In [8]:
df_limpo = df_bruto.copy()

# 1. Para as colunas críticas, criamos a FLAG que guarda o comportamento anômalo
colunas_criticas = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp']

for col in colunas_criticas:
    # Cria uma nova coluna: 1 se for nulo, 0 se estiver preenchido
    df_limpo[f'{col}_isNull'] = df_limpo[col].isnull().astype(int)
    

# 2. Para o grupo de cupons e pedidos (Alerta Verde), mantemos o 0 porque é lógico
df_limpo['CouponUsed'] = df_limpo['CouponUsed'].fillna(0)
df_limpo['OrderCount'] = df_limpo['OrderCount'].fillna(0)
df_limpo['OrderAmountHikeFromlastYear'] = df_limpo['OrderAmountHikeFromlastYear'].fillna(0)

# 3. Para o ruído aleatório, apenas a mediana
df_limpo['DaySinceLastOrder'] = df_limpo['DaySinceLastOrder'].fillna(df_limpo['DaySinceLastOrder'].median())

- Partindo agora pra correção das colunas categoricas que apresentam informações redundantes

In [9]:
# Criando os dicionários de mapeamento com base no seu diagnóstico
map_login_device = {
    'Phone': 'Mobile Phone'
}

map_payment_mode = {
    'CC': 'Credit Card',
    'COD': 'Cash on Delivery'
}

map_order_cat = {
    'Mobile': 'Mobile Phone'
}

# Aplicando as substituições diretamente no df_limpo
df_limpo['PreferredLoginDevice'] = df_limpo['PreferredLoginDevice'].replace(map_login_device)
df_limpo['PreferredPaymentMode'] = df_limpo['PreferredPaymentMode'].replace(map_payment_mode)
df_limpo['PreferedOrderCat'] = df_limpo['PreferedOrderCat'].replace(map_order_cat)

In [10]:
# Verificando os novos resultados agrupados
colunas_ajustadas = ['PreferredLoginDevice', 'PreferredPaymentMode', 'PreferedOrderCat']

for col in colunas_ajustadas:
    print(f"\n--- Valores únicos REAIS na coluna: {col} ---")
    print(df_limpo[col].value_counts())


--- Valores únicos REAIS na coluna: PreferredLoginDevice ---
PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64

--- Valores únicos REAIS na coluna: PreferredPaymentMode ---
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64

--- Valores únicos REAIS na coluna: PreferedOrderCat ---
PreferedOrderCat
Mobile Phone          2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: count, dtype: int64


## Exportação dos dados tratados pro BigQuery:

In [ ]:
ID_PROJETO = "groovy-treat-471720-n1"          # O ID do seu projeto no Google Cloud
ID_DATASET = "1"    # O nome do seu Dataset no BigQuery
NOME_TABELA = "EComm_tratada"    # O nome da nova tabela que será criada

tabela_destino = f"{ID_DATASET}.{NOME_TABELA}"

print(f"Iniciando a exportação para {tabela_destino}...")

# Exportando o DataFrame
df_limpo.to_gbq(
    destination_table=tabela_destino,
    project_id=ID_PROJETO,
    if_exists="replace",  
    progress_bar=True     
)

print(f"Sucesso! Tabela '{NOME_TABELA}' exportada.")